In [ ]:
import pandas as pd
import os
import re
import json
from openai import OpenAI
from google.colab import userdata

# ---------------- CONFIG ----------------
or_token = userdata.get('OR_Key_2')
MODEL = "openai/gpt-5.2"
#OUTPUT_CSV = "outputs_evaluated.csv"

API_KEY = or_token

clientAI = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=or_token,
)


In [ ]:
# ---------------- HELPER FUNCTIONS ----------------
def call_llm(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response = clientAI.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        top_p=1.0
    )
    return response.choices[0].message.content.strip()

def parse_json_like(text):
    """
    Extract the first JSON object found in text.
    """
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("No JSON object found")

    json_str = match.group(0)
    return json.loads(json_str)

# ---------------- PROMPT FUNCTION ----------------
def get_advice_judge_prompts(context, advice):

    system_prompt = (
        "You are an expert rice agronomist acting as an evaluator.\n\n"
        "IMPORTANT EVALUATION RULES:\n"
        "The provided context contains multiple types of information "
        "(symptoms, damage, transmission, and management).\n\n"
        "When evaluating the ADVICE:\n"
        "  - Consider ONLY information related to management practices, control measures, "
        "and local agronomic guidelines.\n"
        "  - Mark hallucination if the advice includes any management action that is NOT "
        "supported by the context.\n\n"
        "Use the definition exactly as provided.\n"
        "Do NOT explain your decision.\n"
        "Output ONLY the requested JSON object.\n\n"
        "Score Definitions:\n"
        "Advice Hallucination (0/1):\n"
        "0 = All management actions are supported by the context.\n"
        "1 = Contains at least one unsupported management action."
    )

    user_prompt = f"""
              Context:
              {context}

              Advice:
              {advice}

              Task:
              Based on the above definitions, return ONLY a JSON object in this format:

              {{
                "advice_hallucination": 0
              }}

              or

              {{
                "advice_hallucination": 1
              }}
              """

    return system_prompt, user_prompt



In [ ]:
import pandas as pd
import os
# ---------------- EXAMPLE USAGE ----------------
# Read input CSV outside
input_file = "/content/evaluation_samples2.csv"
df = pd.read_csv(input_file)

output_csv = "llm_as_judge_output_advice.csv"

In [ ]:
# Loop over rows for advice
for idx, row in df.iterrows():

    file_name = row["file_name"]
    context = row["context"]
    advice = row["reasoning_advice"]

    # Generate prompts for advice evaluation
    sys_prompt, user_prompt = get_advice_judge_prompts(
        context,
        advice
    )

    # Call LLM judge
    judge_output = call_llm(sys_prompt, user_prompt)

    # Parse JSON output
    judge_json = parse_json_like(judge_output)

    # Convert string score to int
    for k in judge_json:
        judge_json[k] = int(judge_json[k])

    # Combine original row + judge result
    combined = row.to_dict()
    combined.update(judge_json)

    combined_df = pd.DataFrame([combined])

    # Write or append
    if not os.path.exists(output_csv):
        combined_df.to_csv(output_csv, index=False)
    else:
        combined_df.to_csv(output_csv, mode="a", header=False, index=False)

    print(f"Processed row {idx} ({file_name}) → appended to {output_csv}")

Processed row 0 (BLB_IMG_7749.JPG) → appended to llm_as_judge_output_advice.csv
Processed row 1 (BLB_8851.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 2 (BLB_3184.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 3 (BLB_8916.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 4 (BLB_IMG_20220407_073928.jpg) → appended to llm_as_judge_output_advice.csv
Processed row 5 (BS_9174.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 6 (BS_IMG_20221106_165849.jpg) → appended to llm_as_judge_output_advice.csv
Processed row 7 (BS_IMG-20221108-WA0061.jpg) → appended to llm_as_judge_output_advice.csv
Processed row 8 (BS_7959.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 9 (BS_IMG_20221106_165513.jpg) → appended to llm_as_judge_output_advice.csv
Processed row 10 (FS_8590.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 11 (FS_156.jpeg) → appended to llm_as_judge_output_advice.csv
Processed row 12 (FS_11

In [ ]:
def get_rationale_judge_prompts(context, rationale):

    system_prompt = (
        "You are an expert rice agronomist acting as an evaluator.\n\n"
        "IMPORTANT EVALUATION RULES:\n"
        "The provided context contains multiple types of information "
        "(symptoms, damage, transmission, and management).\n\n"
        "When evaluating the RATIONALE:\n"
        "  - Consider ONLY information related to the diagnosis, including observed symptoms, "
        "farmer responses, and supporting evidence.\n"
        "  - Mark hallucination if the rationale contains any statement that is NOT "
        "supported by the context.\n\n"
        "Use the definition exactly as provided.\n"
        "Do NOT explain your decision.\n"
        "Output ONLY the requested JSON object.\n\n"
        "Score Definitions:\n"
        "Rationale Hallucination (0/1):\n"
        "0 = Every statement in the rationale is supported by the context.\n"
        "1 = Contains at least one unsupported statement."
    )

    user_prompt = f"""
      Context:
      {context}

      Rationale:
      {rationale}

      Task:
      Based on the above definition, return ONLY a JSON object in this format:

      {{
        "rationale_hallucination": 0
      }}

      or

      {{
        "rationale_hallucination": 1
      }}
      """

    return system_prompt, user_prompt

In [ ]:
import pandas as pd
import os
# ---------------- EXAMPLE USAGE ----------------
# Read input CSV outside
input_file = "/content/evaluation_samples2.csv"
df = pd.read_csv(input_file)

output_csv = "llm_as_judge_output_rationale.csv"

In [ ]:
# Loop over rows
for idx, row in df.iterrows():

    file_name = row["file_name"]
    context = row["context"]
    rationale = row["processed_reasoning"]

    # Generate prompts for rationale evaluation
    sys_prompt, user_prompt = get_rationale_judge_prompts(
        context,
        rationale
    )

    # Call LLM judge
    judge_output = call_llm(sys_prompt, user_prompt)

    # Parse JSON output
    judge_json = parse_json_like(judge_output)

    # Convert string score to int
    for k in judge_json:
        judge_json[k] = int(judge_json[k])

    # Combine original row + judge result
    combined = row.to_dict()
    combined.update(judge_json)

    combined_df = pd.DataFrame([combined])

    # Write or append
    if not os.path.exists(output_csv):
        combined_df.to_csv(output_csv, index=False)
    else:
        combined_df.to_csv(output_csv, mode="a", header=False, index=False)

    print(f"Processed row {idx} ({file_name}) → appended to {output_csv}")

Processed row 0 (BLB_IMG_7749.JPG) → appended to llm_as_judge_output_rationale.csv
Processed row 1 (BLB_8851.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 2 (BLB_3184.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 3 (BLB_8916.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 4 (BLB_IMG_20220407_073928.jpg) → appended to llm_as_judge_output_rationale.csv
Processed row 5 (BS_9174.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 6 (BS_IMG_20221106_165849.jpg) → appended to llm_as_judge_output_rationale.csv
Processed row 7 (BS_IMG-20221108-WA0061.jpg) → appended to llm_as_judge_output_rationale.csv
Processed row 8 (BS_7959.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 9 (BS_IMG_20221106_165513.jpg) → appended to llm_as_judge_output_rationale.csv
Processed row 10 (FS_8590.jpeg) → appended to llm_as_judge_output_rationale.csv
Processed row 11 (FS_156.jpeg) → appended to llm_as_judge_output_r